# Teaching Effectiveness Assessment Tool
## Phase 1 — Training Pipeline

**Modernized from BSU ECE Capstone 2023 (Romasanta et al.)**

This notebook walks through the full training pipeline:
1. Load MUD Card dataset
2. Store raw data in SQLite
3. Translate Tagalog → English
4. Label comments (positive / negative)
5. Clean text
6. Split and vectorize
7. Train Multinomial Naive Bayes
8. Evaluate (accuracy + confusion matrix)
9. Save model and vectorizer

## Setup

In [ ]:
import os, re, string, sqlite3, pickle, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt

from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

# Paths
DATA_PATH   = 'data/MUD_Card_Data.tsv'
DB_PATH     = 'data/training.db'
MODEL_PATH  = '../models/sentiment_model.sav'
VECTOR_PATH = '../models/tfidfvectorizer.sav'

print('Setup complete.')

## Step 1 — Load MUD Card Dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH, delimiter='\t')
print(f'Loaded {len(df_raw)} rows')
df_raw.head()

## Step 2 — Store Raw Data in SQLite

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_raw.to_sql('raw_mudcard', conn, if_exists='replace', index=False)
conn.close()
print(f'Raw data stored in SQLite → {DB_PATH}')

## Step 3 — Translate Tagalog → English

In [ ]:
def translate_text(text):
    try:
        if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
            return ''
        if detect(text) == 'tl':
            return GoogleTranslator(source='tl', target='en').translate(text)
        return text
    except (LangDetectException, Exception):
        return str(text) if text else ''

df_translated = df_raw.copy()
print('Translating Positive Comments...')
df_translated['Positive Comments'] = df_translated['Positive Comments'].apply(translate_text)
print('Translating Negative Comments...')
df_translated['Negative Comments'] = df_translated['Negative Comments'].apply(translate_text)

conn = sqlite3.connect(DB_PATH)
df_translated.to_sql('translated_mudcard', conn, if_exists='replace', index=False)
conn.close()

print('Translation complete. Saved → translated_mudcard')
df_translated.head()

## Step 4 — Label Comments (Positive / Negative)

In [ ]:
features, sentiments = [], []

for text in df_translated['Negative Comments']:
    t = str(text).strip().lower()
    if t and t not in ('none', 'n/a', 'nan'):
        features.append(str(text))
        sentiments.append('negative')

for text in df_translated['Positive Comments']:
    t = str(text).strip().lower()
    if t and t not in ('none', 'n/a', 'nan'):
        features.append(str(text))
        sentiments.append('positive')

df_labeled = pd.DataFrame({'Features': features, 'Sentiments': sentiments})

conn = sqlite3.connect(DB_PATH)
df_labeled.to_sql('labeled_comments', conn, if_exists='replace', index=False)
conn.close()

print(f'Total: {len(df_labeled)} | Positive: {(df_labeled.Sentiments=="positive").sum()} | Negative: {(df_labeled.Sentiments=="negative").sum()}')
df_labeled.head()

## Step 5 — Clean Text

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation + '*_'), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'[\u2018\u2019\u201c\u201d\u2026]', '', text)
    tokens = word_tokenize(text)
    tokens = [t.translate(str.maketrans('', '', string.punctuation))
              for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

df_cleaned = df_labeled.copy()
df_cleaned['Features'] = df_cleaned['Features'].apply(clean_text)
df_cleaned = df_cleaned[df_cleaned['Features'].str.strip() != ''].reset_index(drop=True)

conn = sqlite3.connect(DB_PATH)
df_cleaned.to_sql('cleaned_comments', conn, if_exists='replace', index=False)
conn.close()

print(f'Cleaned — {len(df_cleaned)} rows remaining')
df_cleaned.head()

## Step 6 — Data Splitting and TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_cleaned['Features'], df_cleaned['Sentiments'],
    test_size=0.2, random_state=143
)

vectorizer = TfidfVectorizer(max_features=1000, stop_words=list(STOP_WORDS))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')

## Step 7 — Train Multinomial Naive Bayes

In [ ]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)
print('Model trained.')

## Step 8 — Evaluate Model

In [ ]:
train_pred = model.predict(X_train_vec)
test_pred  = model.predict(X_test_vec)

train_acc = metrics.accuracy_score(y_train, train_pred)
test_acc  = metrics.accuracy_score(y_test, test_pred)

print(f'Training Accuracy: {train_acc*100:.2f}%')
print(classification_report(y_train, train_pred, digits=3))
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(classification_report(y_test, test_pred, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (preds, labels, title) in zip(axes, [
    (train_pred, y_train, 'Training Confusion Matrix'),
    (test_pred,  y_test,  'Test Confusion Matrix'),
]):
    cm = confusion_matrix(labels, preds, labels=model.classes_)
    ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Step 9 — Save Model and Vectorizer

In [ ]:
os.makedirs('../models', exist_ok=True)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
with open(VECTOR_PATH, 'wb') as f:
    pickle.dump(vectorizer, f)

print(f'Model saved     → {MODEL_PATH}')
print(f'Vectorizer saved → {VECTOR_PATH}')
print(f'\nTraining complete! Test Accuracy: {test_acc*100:.2f}%')